In [ ]:
!apt-get install openjdk-8-jdk-headless #jdk install
!wget -q http://archive.apache.org/dist/spark/spark-3.0.0/spark-3.0.0-bin-hadoop3.2.tgz #spark file
!tar -xf spark-3.0.0-bin-hadoop3.2.tgz
!pip install findspark

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libxtst6 openjdk-8-jre-headless
Suggested packages:
  openjdk-8-demo openjdk-8-source libnss-mdns fonts-dejavu-extra fonts-nanum fonts-ipafont-gothic
  fonts-ipafont-mincho fonts-wqy-microhei fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  libxtst6 openjdk-8-jdk-headless openjdk-8-jre-headless
0 upgraded, 3 newly installed, 0 to remove and 49 not upgraded.
Need to get 39.6 MB of archives.
After this operation, 144 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libxtst6 amd64 2:1.2.3-1build4 [13.4 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 openjdk-8-jre-headless amd64 8u422-b05-1~22.04 [30.8 MB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 openjdk-8-jdk-headless amd64 8u422-b05-1~22.04 [8,843 kB]
Fetched 39.

In [ ]:
import os
import findspark

#환경변수에 path 지정
os.environ['JAVA_HOME'] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ['SPARK_HOME'] = '/content/spark-3.0.0-bin-hadoop3.2'

#spark의 경우 잘 찾지 못하는 경우가 있어 findspark를 이용
findspark.init()

In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark.sql import types as T
from pyspark.sql import window as W
from pyspark.sql import functions as F

In [ ]:
spark = SparkSession.builder.master("local").appName("Colab").getOrCreate()

In [ ]:
os.getcwd()

'/content'

## 1. member 테이블에서 회원 상태별 인원수를 내림차순으로 보여주세요.

In [ ]:
mdf = spark.read.parquet('./sample_data/member.parquet', header=True)
mdf.show(10)
mdf.printSchema()

+------+---+--------+-------+
|   idx|sex|  status|  grade|
+------+---+--------+-------+
|   100| 남|유료회원|초1학년|
|  1000| 여|유료회원|초5학년|
| 10000| 여|유료회원|초6학년|
|100007| 남|유료회원|초4학년|
| 10001| 남|유료회원|초2학년|
|100010| 남|유료회원|초4학년|
|100011| 여|유료회원|초4학년|
|100017| 여|유료회원|  0학년|
|100019| 남|유료회원|초3학년|
| 10002| 여|유료회원|초4학년|
+------+---+--------+-------+
only showing top 10 rows

root
 |-- idx: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- status: string (nullable = true)
 |-- grade: string (nullable = true)



In [ ]:
res = mdf.groupBy("status").agg(F.count("status").alias("count")).orderBy(F.col("count").desc())

In [ ]:
res.show()

+--------+-----+
|  status|count|
+--------+-----+
|유료회원|66032|
|학습만료|   92|
|    신규|   75|
|  재구매|   70|
|    이월|   22|
|    복회|    7|
|    취소|    6|
+--------+-----+



## 2. study_his 테이블의 pointnm 컬럼에 대해 공백을 모두 없애주세요.


In [ ]:
sdf = spark.read.parquet("./sample_data/study_his.parquet", header=True)

sdf.show(5)
sdf.printSchema()

+-----+-------+--------+-----------+
|  idx|proc_ym|proc_ymd|    pointnm|
+-----+-------+--------+-----------+
|88311| 202306|20230628|한글 스피치|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
+-----+-------+--------+-----------+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- proc_ym: string (nullable = true)
 |-- proc_ymd: string (nullable = true)
 |-- pointnm: string (nullable = true)



In [ ]:
res = sdf.withColumn("pointnm", F.regexp_replace(F.col("pointnm"), " ", ""))

In [ ]:
res.show(5)

+-----+-------+--------+----------+
|  idx|proc_ym|proc_ymd|   pointnm|
+-----+-------+--------+----------+
|88311| 202306|20230628|한글스피치|
| 8604| 202306|20230606| 중학3학년|
| 8604| 202306|20230606| 중학3학년|
| 8604| 202306|20230606| 중학3학년|
| 8604| 202306|20230606| 중학3학년|
+-----+-------+--------+----------+
only showing top 5 rows



## 3. point_his의 proc_ymd 컬럼의 날짜 표현 형식을 yyyy-mm-dd 형식이 되도록 바꿔주세요.(UDF, slicing 사용 X)

In [ ]:
pdf = spark.read.parquet("./sample_data/point_his.parquet", header=True)

pdf.show(5)
pdf.printSchema()

+-----+-------+--------+-----+
|  idx|proc_ym|proc_ymd|point|
+-----+-------+--------+-----+
|96465| 202306|20230624| 1000|
|96465| 202306|20230624|  500|
|87940| 202304|20230405| 2000|
|87940| 202304|20230405| 3500|
|87940| 202304|20230405| 4000|
+-----+-------+--------+-----+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- proc_ym: string (nullable = true)
 |-- proc_ymd: string (nullable = true)
 |-- point: string (nullable = true)



In [ ]:
res = pdf.withColumn("proc_ymd", F.date_format(F.to_date(F.col("proc_ymd"), "yyyyMMdd"), "yyyy-MM-dd"))

In [ ]:
res.show(5)

+-----+-------+----------+-----+
|  idx|proc_ym|  proc_ymd|point|
+-----+-------+----------+-----+
|96465| 202306|2023-06-24| 1000|
|96465| 202306|2023-06-24|  500|
|87940| 202304|2023-04-05| 2000|
|87940| 202304|2023-04-05| 3500|
|87940| 202304|2023-04-05| 4000|
+-----+-------+----------+-----+
only showing top 5 rows



## 4. member 회원상태(status)가 학습만료 회원들 중 포인트가 가장 많은 TOP3 학생을 뽑아주세요.

In [ ]:
mdf.show(5)
mdf.printSchema()

+------+---+--------+-------+
|   idx|sex|  status|  grade|
+------+---+--------+-------+
|   100| 남|유료회원|초1학년|
|  1000| 여|유료회원|초5학년|
| 10000| 여|유료회원|초6학년|
|100007| 남|유료회원|초4학년|
| 10001| 남|유료회원|초2학년|
+------+---+--------+-------+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- status: string (nullable = true)
 |-- grade: string (nullable = true)



In [ ]:
new_df1 = mdf.join(pdf, mdf.idx == pdf.idx, "outer")

In [ ]:
new_df1.show(10)

+------+---+--------+-------+----+-------+--------+-----+
|   idx|sex|  status|  grade| idx|proc_ym|proc_ymd|point|
+------+---+--------+-------+----+-------+--------+-----+
|100010| 남|유료회원|초4학년|null|   null|    null| null|
|100140| 여|유료회원|초4학년|null|   null|    null| null|
| 10096| 여|  재구매|초5학년|null|   null|    null| null|
| 10436| 남|유료회원|초6학년|null|   null|    null| null|
| 11078| 여|유료회원|초3학년|null|   null|    null| null|
| 11332| 남|유료회원|초2학년|null|   null|    null| null|
| 11563| 여|유료회원|초3학년|null|   null|    null| null|
|  1159| 남|유료회원|초5학년|null|   null|    null| null|
| 11722| 남|유료회원|초2학년|null|   null|    null| null|
| 11888| 남|유료회원|초2학년|null|   null|    null| null|
+------+---+--------+-------+----+-------+--------+-----+
only showing top 10 rows



In [ ]:
mdf.filter(F.col("status") == "학습만료").show(5)

+-----+---+--------+-------+
|  idx|sex|  status|  grade|
+-----+---+--------+-------+
|10013| 남|학습만료|초3학년|
|10665| 남|학습만료|초3학년|
|11280| 남|학습만료|초5학년|
|11286| 여|학습만료|초3학년|
|11710| 여|학습만료|초3학년|
+-----+---+--------+-------+
only showing top 5 rows



In [ ]:
new_df1.filter((F.col("status") == "학습만료") & (F.col("point") > 0)).show()

+---+---+------+-----+---+-------+--------+-----+
|idx|sex|status|grade|idx|proc_ym|proc_ymd|point|
+---+---+------+-----+---+-------+--------+-----+
+---+---+------+-----+---+-------+--------+-----+



In [ ]:
new_df1 = (mdf.join(pdf, mdf.idx == pdf.idx, "outer").select(mdf["idx"], "status", "point"))

In [ ]:
_window = W.Window.partitionBy("status").orderBy(F.col("total_point").desc())

In [ ]:
filtered_df = new_df1.filter(F.col("status") == "학습만료")

In [ ]:
ranked_df = filtered_df.groupBy("idx", "status").agg(F.sum("point").alias("total_point")).withColumn("rank", F.rank().over(_window))

In [ ]:
ranked_df.filter(F.col("total_point") > 0).show()

+---+------+-----------+----+
|idx|status|total_point|rank|
+---+------+-----------+----+
+---+------+-----------+----+



In [ ]:
top3_df = ranked_df.filter(F.col("rank") <= 3)

In [ ]:
top3_df.show(5)

+-----+--------+-----------+----+
|  idx|  status|total_point|rank|
+-----+--------+-----------+----+
|11710|학습만료|       null|   1|
| 4109|학습만료|       null|   1|
|52662|학습만료|       null|   1|
|70259|학습만료|       null|   1|
|59237|학습만료|       null|   1|
+-----+--------+-----------+----+
only showing top 5 rows



## 5. member_dup 테이블에 학년이 여러개인 idx가 있습니다. 해당 idx의 학년 중 가장 높은 학년만 남겨서 idx와 grade가 1:1 대응이 되도록 만들어주세요.

In [ ]:
mem_d_df = spark.read.parquet("./sample_data/member_dup.parquet", header=True)

mem_d_df.show(5)
mem_d_df.printSchema()

+-----+---+--------+-------+
|  idx|sex|  status|  grade|
+-----+---+--------+-------+
| 6884| 여|유료회원|초3학년|
| 6331| 남|유료회원|초3학년|
|69294| 남|유료회원|초5학년|
|31531| 여|유료회원|초1학년|
|85784| 여|유료회원|초2학년|
+-----+---+--------+-------+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- status: string (nullable = true)
 |-- grade: string (nullable = true)



In [ ]:
res = mem_d_df.groupBy("idx").agg(F.max("grade").alias("grade"))

In [ ]:
res.show(5)

+------+-------+
|   idx|  grade|
+------+-------+
|100010|초4학년|
|100140|초4학년|
| 10096|초5학년|
| 10436|초6학년|
| 11078|초4학년|
+------+-------+
only showing top 5 rows



## 6. regdate_his 등록일(reg_date)이 2023.03.15일인 유료회원들이 가장 많이 착용한 아이템(codename) TOP3를 뽑아주세요.

In [ ]:
rdf = spark.read.parquet("./sample_data/regdate.parquet", header=True)

rdf.show(5)
rdf.printSchema()

+---+--------+
|idx| regdate|
+---+--------+
|  1|20221206|
|  2|20221206|
|  3|20221206|
|  4|20221206|
|  5|20221206|
+---+--------+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- regdate: string (nullable = true)



In [ ]:
idf = spark.read.parquet("./sample_data/item_his.parquet", header=True)

idf.show(5)
idf.printSchema()

+-----+-----+-------+--------+----------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+-----+-------+--------+----------+--------------+-----+
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
+-----+-----+-------+--------+----------+--------------+-----+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- lv: string (nullable = true)
 |-- proc_ym: string (nullable = true)
 |-- proc_ymd: string (nullable = true)
 |-- codename: string (nullable = true)
 |-- mascodename: string (nullable = true)
 |-- price: string (nullable = true)



In [ ]:
join1 = idf.join(rdf, idf.idx == rdf.idx, "left").select(idf["idx"], "regdate", "codename")

In [ ]:
join_df = join1.join(mdf, join1.idx == mdf.idx, "left").select(join1["idx"], "regdate", "codename", "status")

In [ ]:
join_df.show(5)

+-----+--------+--------+--------+
|  idx| regdate|codename|  status|
+-----+--------+--------+--------+
|53687|20230508|액세서리|유료회원|
|53687|20230508|액세서리|유료회원|
|53687|20230508|액세서리|유료회원|
|53687|20230508|액세서리|유료회원|
|53687|20230508|액세서리|유료회원|
+-----+--------+--------+--------+
only showing top 5 rows



In [ ]:
join_df = join_df.filter((F.col("regdate") == "20230315") & (F.col("status") == "유료회원"))

In [ ]:
join_df.show(5)

+-----+--------+----------+--------+
|  idx| regdate|  codename|  status|
+-----+--------+----------+--------+
|84662|20230315|      하의|유료회원|
|84662|20230315|      헤어|유료회원|
|84664|20230315|상태메시지|유료회원|
|84664|20230315|상태메시지|유료회원|
|84664|20230315|상태메시지|유료회원|
+-----+--------+----------+--------+
only showing top 5 rows



In [ ]:
_window = W.Window.orderBy(F.col("count").desc())

In [ ]:
group_df = join_df.groupBy("codename").count().withColumn("rank", F.rank().over(_window))

In [ ]:
res = group_df.filter(F.col("rank") <= 3)

In [ ]:
res.show(5)

+----------+-----+----+
|  codename|count|rank|
+----------+-----+----+
|상태메시지|   74|   1|
|      헤어|   67|   2|
|      얼굴|   47|   3|
+----------+-----+----+



## 7. member 테이블의 grade 컬럼에서 숫자만 뽑아 grade라는 컬럼을 재구성해주세요. (초3학년 -> 3) (UDF, slicing 사용 X)

In [ ]:
mdf.show(5)
mdf.printSchema()

+------+---+--------+-------+
|   idx|sex|  status|  grade|
+------+---+--------+-------+
|   100| 남|유료회원|초1학년|
|  1000| 여|유료회원|초5학년|
| 10000| 여|유료회원|초6학년|
|100007| 남|유료회원|초4학년|
| 10001| 남|유료회원|초2학년|
+------+---+--------+-------+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- status: string (nullable = true)
 |-- grade: string (nullable = true)



In [ ]:
res = mdf.withColumn("grade", F.regexp_extract(F.col("grade"), r"\d+", 0))

In [ ]:
res.show(5)

+------+---+--------+-----+
|   idx|sex|  status|grade|
+------+---+--------+-----+
|   100| 남|유료회원|    1|
|  1000| 여|유료회원|    5|
| 10000| 여|유료회원|    6|
|100007| 남|유료회원|    4|
| 10001| 남|유료회원|    2|
+------+---+--------+-----+
only showing top 5 rows



## 8. study_his 월별로 pointnm 컬럼에 대한 point 발생 수(count)를 오름차순으로 순번 매겨주세요.

In [ ]:
sdf.show(5)
sdf.printSchema()

+-----+-------+--------+-----------+
|  idx|proc_ym|proc_ymd|    pointnm|
+-----+-------+--------+-----------+
|88311| 202306|20230628|한글 스피치|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
+-----+-------+--------+-----------+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- proc_ym: string (nullable = true)
 |-- proc_ymd: string (nullable = true)
 |-- pointnm: string (nullable = true)



In [ ]:
sdf = sdf.withColumn("month", F.substring(F.col("proc_ym"), -1, 1))

In [ ]:
sdf.show(5)

+-----+-------+--------+-----------+-----+
|  idx|proc_ym|proc_ymd|    pointnm|month|
+-----+-------+--------+-----------+-----+
|88311| 202306|20230628|한글 스피치|    6|
| 8604| 202306|20230606| 중학 3학년|    6|
| 8604| 202306|20230606| 중학 3학년|    6|
| 8604| 202306|20230606| 중학 3학년|    6|
| 8604| 202306|20230606| 중학 3학년|    6|
+-----+-------+--------+-----------+-----+
only showing top 5 rows



In [ ]:
group_sdf = sdf.groupBy("month", "pointnm").agg(F.count(F.col("month")).alias("count"))

In [ ]:
_window = W.Window.partitionBy("month").orderBy(F.col("count").asc())

In [ ]:
ranked_sdf = group_sdf.withColumn("rank", F.rank().over(_window))

In [ ]:
ranked_sdf.show(5)

+-----+----------------+-----+----+
|month|         pointnm|count|rank|
+-----+----------------+-----+----+
|    5|       학교 체험|    7|   1|
|    5|        받아쓰기|   42|   2|
|    5|       중학 특강|   42|   2|
|    5|학교 공부 맛보기|   99|   4|
|    5|         AI 국어|  182|   5|
+-----+----------------+-----+----+
only showing top 5 rows



## 9. 레벨이 151~160 에 있는 유저들 중 딱 한 명씩만 등록한 날짜들을 구해주세요. 레벨은 idx가 가진 레벨중 가장 높은 레벨로 사용(10) -> 중복처리에 유의

In [ ]:
idf.show(5)
idf.printSchema()

+-----+-----+-------+--------+----------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+-----+-------+--------+----------+--------------+-----+
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
+-----+-----+-------+--------+----------+--------------+-----+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- lv: string (nullable = true)
 |-- proc_ym: string (nullable = true)
 |-- proc_ymd: string (nullable = true)
 |-- codename: string (nullable = true)
 |-- mascodename: string (nullable = true)
 |-- price: string (nullable = true)



In [ ]:
join1 = idf.join(rdf, idf.idx == rdf.idx, "left").select(idf["idx"], "regdate", "lv")

In [ ]:
filtered_idf = join1.filter((F.col("lv") >= 151) & (F.col("lv") <= 160))

In [ ]:
_window = W.Window.partitionBy("idx").orderBy(F.col("lv").desc())

In [ ]:
h_lv_df = filtered_idf.withColumn("rank", F.rank().over(_window)).filter(F.col("rank") == 1)

In [ ]:
dist_df = h_lv_df.groupBy("regdate").count()

In [ ]:
dist_df.filter(F.col("count") == 1).show()

+--------+
| regdate|
+--------+
|20230127|
|20230203|
|20230101|
|20230105|
|20230201|
+--------+
only showing top 5 rows



## 10. item_his 테이블에서 레벨이 null인 유저가 가장 많은 날짜를 구해주세요.

In [ ]:
idf.show(5)
idf.printSchema()

+-----+-----+-------+--------+----------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+-----+-------+--------+----------+--------------+-----+
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
+-----+-----+-------+--------+----------+--------------+-----+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- lv: string (nullable = true)
 |-- proc_ym: string (nullable = true)
 |-- proc_ymd: string (nullable = true)
 |-- codename: string (nullable = true)
 |-- mascodename: string (nullable = true)
 |-- price: string (nullable = true)



In [ ]:
idf_n = idf.filter(F.col("lv").isNull())

In [ ]:
idf_n.show(5)

+-----+----+-------+--------+--------+--------------+-----+
|  idx|  lv|proc_ym|proc_ymd|codename|   mascodename|price|
+-----+----+-------+--------+--------+--------------+-----+
|92027|null| 202305|20230505|  코스튬|아바타파츠구분|  300|
|92131|null| 202305|20230507|  코스튬|아바타파츠구분|  300|
|91962|null| 202305|20230506|  코스튬|아바타파츠구분|  300|
|94835|null| 202306|20230605|  코스튬|아바타파츠구분|  300|
|94835|null| 202306|20230605|  코스튬|아바타파츠구분|  300|
+-----+----+-------+--------+--------+--------------+-----+
only showing top 5 rows



In [ ]:
res = idf_n.groupBy("proc_ymd").count().orderBy(F.col("count").desc())

In [ ]:
res.first().proc_ymd

'20230504'

## 11. pointnm 별로 획득한 point의 종류가 언더바(_)로 이어져서 보이도록 테이블을 만들어 주세요.

In [ ]:
pdf.show(5)
pdf.printSchema()

+-----+-------+--------+-----+
|  idx|proc_ym|proc_ymd|point|
+-----+-------+--------+-----+
|96465| 202306|20230624| 1000|
|96465| 202306|20230624|  500|
|87940| 202304|20230405| 2000|
|87940| 202304|20230405| 3500|
|87940| 202304|20230405| 4000|
+-----+-------+--------+-----+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- proc_ym: string (nullable = true)
 |-- proc_ymd: string (nullable = true)
 |-- point: string (nullable = true)



In [ ]:
join_df = pdf.join(sdf, pdf.proc_ymd == sdf.proc_ymd, "inner").select("point", "pointnm")

In [ ]:
join_df.show(5)

+-----+-----------+
|point|    pointnm|
+-----+-----------+
| 3500|한글 스피치|
| 4000|한글 스피치|
| 4500|한글 스피치|
| 2500|한글 스피치|
| 3000|한글 스피치|
+-----+-----------+
only showing top 5 rows



In [ ]:
grouped_df = join_df.groupBy("pointnm").agg(F.collect_list("point").alias("points"))

In [ ]:
@F.udf(returnType=T.StringType())
def concat_points(points):
  return "_".join(str(x) for x in points)

In [ ]:
res = grouped_df.withColumn("points", concat_points(F.col("points")))

In [ ]:
# res.show(5)  # point 종류 컬럼이 어떤 건지 모르겠음

KeyboardInterrupt: 

## 12. member의 status 컬럼에서 '회원'단어가 있으면 띄어쓰기하고 없으면 띄어쓰기 후 회원 단어 추가해주세요. (e.g. 유료회원 -> 유효 회원 / 신규 -> 신규 회원)

In [ ]:
mdf.show(5)
mdf.printSchema()

+------+---+--------+-------+
|   idx|sex|  status|  grade|
+------+---+--------+-------+
|   100| 남|유료회원|초1학년|
|  1000| 여|유료회원|초5학년|
| 10000| 여|유료회원|초6학년|
|100007| 남|유료회원|초4학년|
| 10001| 남|유료회원|초2학년|
+------+---+--------+-------+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- status: string (nullable = true)
 |-- grade: string (nullable = true)



In [ ]:
@F.udf(returnType=T.StringType())
def add_word(status):
  if "회원" in status:
    return status.replace("회원", " 회원")
  else:
    return status + " 회원"

In [ ]:
res = mdf.withColumn("status", add_word(F.col("status")))

In [ ]:
res.printSchema()

root
 |-- idx: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- status: string (nullable = true)
 |-- grade: string (nullable = true)



In [ ]:
res.show(5)

+------+---+---------+-------+
|   idx|sex|   status|  grade|
+------+---+---------+-------+
|   100| 남|유료 회원|초1학년|
|  1000| 여|유료 회원|초5학년|
| 10000| 여|유료 회원|초6학년|
|100007| 남|유료 회원|초4학년|
| 10001| 남|유료 회원|초2학년|
+------+---+---------+-------+
only showing top 5 rows

